The only important variables to set in this entire script are found in the second cell, and are named:

1. **subject_ID**
2. **user**
3. **execute**

The directory and paths are all set to *Omri's* directory. It doesn't make sense to store the data in two places on Milgram since *Aryan* already has access to data in *Omri's* scratch folder. Thus, these don't need to be changed or manipulated.

Also, the following link is the BIDS specification [https://bids-specification.readthedocs.io/en/stable/modality-specific-files/task-events.html]

It contains the relevant information to specify renaming formats. I am sure of the following:

1. Functional Runs: sub-multimem-p001_task-multisensorymemory_run-1_bold.nii.gz
2. Anatomical Scans: sub-multimem-p001_anat-T1w.nii.gz
3. Events Files: sub-multimem-p001_task-multisensorymemory_run-1_events.csv

Our events files should actually be present in two formats, **.tsv** *(not a typo)* and **.json** (explained in the link).

In [1]:
import os
import json
import shutil
import pandas as pd
import numpy as np
import argparse
import sys
from scipy.io import loadmat

# Set the subject we are renaming as a 3 digit string, 
# Set the user's NET ID
# Only set execute to true if we want to perform the renaming, otherwise only prints calculated names
subject_ID = 'pp_03'
user = "aa2842"
execute = False

######################## No other variables to be hard-coded ########################################################################

# Specify directory with nii files
directory = f'/gpfs/milgram/scratch60/turk-browne/{user}/sandbox/auditory-object-scenes-data/{subject_ID}_nii/'
bids_dir = f'/gpfs/milgram/scratch60/turk-browne/{user}/sandbox/auditory-object-scenes-data/{subject_ID}_nii_bids/'

In [2]:
# Helper function to transform .mat files to TSV files
def transform_mat_to_tsv(path_to_events_file, directory, execute, verbose):
    print(path_to_events_file)
    # read data from nested .mat structure
    if 'visual' in path_to_events_file:
        events = loadmat(file_name=path_to_events_file)
        onset = events['data']['blockOnset'][0][0]
        duration = events['data']['blockDurations'][0][0]
        stimulus_name = events['data']['blockCondition'][0][0]
    elif 'auditory' in path_to_events_file:
        events = loadmat(file_name=path_to_events_file)
        onset = events['data']['Onsets'][0][0]
        duration = events['data']['Durations'][0][0]
        stimulus_name = events['data']['trialType'][0][0]
    else:
        raise ValueError("The given file is neither visual nor auditory - not a valid events .mat")
        
    # convert values to arrays
    onset = np.concatenate(onset).ravel()
    duration = np.concatenate(duration).ravel()
    stimulus_name = np.concatenate(stimulus_name).ravel()

    # create panda with columns onset duration and stimulus name
    events_panda = pd.DataFrame({'onset': onset, 'duration': duration, 'trial_type': stimulus_name})

    # Save the file as a tsv with the same name
    print(f"Ready to save {path_to_events_file} as a tsv")
    if execute:
        temp_path = os.path.join(directory, os.path.basename(path_to_events_file).replace(".mat", ".tsv"))
        events_panda.to_csv(temp_path, sep='\t', index=False)
        print(f"saved to {temp_path}")
    if verbose: print(events_panda)

    return 0

In [38]:
# Convert mat files to tsv and save them
mat_files = [f for f in os.listdir(directory) if f.endswith('.mat')]
for mat_file in mat_files:
    transform_mat_to_tsv(os.path.join(directory, mat_file), directory, execute=True, verbose=False)

/gpfs/milgram/scratch60/turk-browne/aa2842/sandbox/auditory-object-scenes-data/pp_03_nii/events_visual_run_1_sub_pp_03.mat
Ready to save /gpfs/milgram/scratch60/turk-browne/aa2842/sandbox/auditory-object-scenes-data/pp_03_nii/events_visual_run_1_sub_pp_03.mat as a tsv
saved to /gpfs/milgram/scratch60/turk-browne/aa2842/sandbox/auditory-object-scenes-data/pp_03_nii/events_visual_run_1_sub_pp_03.tsv
/gpfs/milgram/scratch60/turk-browne/aa2842/sandbox/auditory-object-scenes-data/pp_03_nii/events_auditory_run_1_sub_pp_03.mat
Ready to save /gpfs/milgram/scratch60/turk-browne/aa2842/sandbox/auditory-object-scenes-data/pp_03_nii/events_auditory_run_1_sub_pp_03.mat as a tsv
saved to /gpfs/milgram/scratch60/turk-browne/aa2842/sandbox/auditory-object-scenes-data/pp_03_nii/events_auditory_run_1_sub_pp_03.tsv
/gpfs/milgram/scratch60/turk-browne/aa2842/sandbox/auditory-object-scenes-data/pp_03_nii/events_auditory_run_3_sub_pp_03.mat
Ready to save /gpfs/milgram/scratch60/turk-browne/aa2842/sandbox/au

In [39]:
# Choose all non '.' files in the directory
files = [f for f in os.listdir(directory) if f.endswith('.json') or f.endswith('.nii') or f.endswith('.csv') or f.endswith('.tsv') or f.endswith('.mat')]
files = [f for f in files if 'practice' not in f and 'order' not in f and 'seq' not in f and 'visual_loc_block' not in f]

# Sort so that we rename nii files first (allows JSON reference with same name)
files = sorted(files, key=lambda x: (x.endswith('.json'), x))

for file in files:
    print(file)

events_auditory_run_1_sub_pp_03.mat
events_auditory_run_1_sub_pp_03.tsv
events_auditory_run_3_sub_pp_03.mat
events_auditory_run_3_sub_pp_03.tsv
events_visual_run_1_sub_pp_03.mat
events_visual_run_1_sub_pp_03.tsv
pp_03_20260415131256_1.nii
pp_03_20260415131256_10.nii
pp_03_20260415131256_11.nii
pp_03_20260415131256_12.nii
pp_03_20260415131256_2.nii
pp_03_20260415131256_3.nii
pp_03_20260415131256_4.nii
pp_03_20260415131256_5.nii
pp_03_20260415131256_6.nii
pp_03_20260415131256_7.nii
pp_03_20260415131256_8.nii
pp_03_20260415131256_9.nii
pp_03_20260415131256_1.json
pp_03_20260415131256_10.json
pp_03_20260415131256_11.json
pp_03_20260415131256_12.json
pp_03_20260415131256_2.json
pp_03_20260415131256_3.json
pp_03_20260415131256_4.json
pp_03_20260415131256_5.json
pp_03_20260415131256_6.json
pp_03_20260415131256_7.json
pp_03_20260415131256_8.json
pp_03_20260415131256_9.json


In [40]:
# Helper function to locate the JSON file corresponding to a CSV events file
def find_json(directory, files, task, run):
    if run == "0":
        print("This is the practice run")
        x = "practice"
    elif task == "auditory":
        x = f"audlocalizer_run-{run}"
    elif task == "visual":
        x = f"vislocalizer"
    else:
        raise ValueError("Wrong task ID given")

    for file in files:
        if file.endswith('.json'):
            with open(os.path.join(directory, file), 'r') as json_file:
                json_data = json.load(json_file)
                if x in json_data['SeriesDescription']:
                    return file
    raise ValueError(f"Wrong task ID – {task} and run – {run} combination")
    return 0

In [41]:
# Example Usage: find the JSON file corresponding to run 1 inside files
find_json(directory, files, "auditory", "01")

'pp_03_20260415131256_7.json'

In [42]:
# This function performs the main renaming of the files when given arguments
# called files and directory, which specify the files to be renamed
def rename(directory, files, execute, verbose):
    names = []
    # loop over every file to be renamed
    for filename in files:
        file_path = os.path.join(directory, filename)

        # This conditional sets the path to reference the .json where the "code" is extracted form
        # The code specifies the type of scan and for nii files we reference the json for extraction

        if filename.endswith('.json'):
            json_path = file_path
        elif filename.endswith('.nii'):
            json_path = os.path.join(directory, f"{filename.split('.')[0]}.json")
        elif filename.endswith('.tsv'):
            run_id = filename.split("_")[3]
            run_id = run_id if run_id == "0" else run_id.zfill(2)
            task_id = filename.split("_")[1]
            json_path = os.path.join(directory, find_json(directory, files, task_id, run_id))
        elif filename.endswith('.mat'):
            run_id = filename.split("_")[3]
            run_id = run_id if run_id == "0" else run_id.zfill(2)
            task_id = filename.split("_")[1]
            json_path = os.path.join(directory, find_json(directory, files, task_id, run_id))

        with open(json_path, 'r') as json_file:
            json_data = json.load(json_file)
            code = json_data['SeriesDescription']
            # This conditional adds the acquisition time to audio test file names so if we repeat then we don't over write files
            if 'audiotest' in code:
                acq_time = f"_{json_data['AcquisitionTime']}"
            else:
                acq_time = ""

        # These conditionals allow for specific formatting (according to BIDS) for the various scans
        if 'T1' in code or 'T2' in code:
            # This is an anatomical run
            sub_id = filename.split('_202')[0]
            new_path = f"sub-{sub_id}_{code}.{filename.split('.')[-1]}"
        elif 'scout' in code:
            # This is a scout run
            sub_id = filename.split('_202')[0]
            new_path = f"sub-{sub_id}_{code}_scout.{filename.split('.')[-1]}"
        elif 'epi' in code:
            # This is a fieldmap
            sub_id = filename.split('_202')[0]
            new_path = f"sub-{sub_id}_{code}.{filename.split('.')[-1]}"
        else:
            # This is a functional run or the .csv file for the run or the .mat file for the run (including audio test)
            if filename.endswith('.nii') or filename.endswith('.json'): # the nii data or jsons
                sub_id = filename.split('_202')[0]
                new_path = f"sub-{sub_id}_{code}_bold{acq_time}.{filename.split('.')[-1]}"
            elif filename.endswith('.tsv') or filename.endswith('.mat'): # events files
                sub_id = json_path.split('/')[-1].split('_202')[0]
                new_path = f"sub-{sub_id}_{code}_events{acq_time}.{filename.split('.')[-1]}"

        # Conditional to rename/print.
        if execute:
            os.rename(file_path, os.path.join(directory, new_path))
            if verbose: print(f"{filename} renamed to {new_path}")
        else:
            if verbose: print(f"{filename} renamed to {new_path}")
        names.append(new_path)
    print("Names have been assigned")

    return names

In [44]:
# Usage with execute set as False
new_names = rename(directory, files, execute=True, verbose=True)

events_auditory_run_1_sub_pp_03.mat renamed to sub-pp_03_audlocalizer_run-01_events.mat
events_auditory_run_1_sub_pp_03.tsv renamed to sub-pp_03_audlocalizer_run-01_events.tsv
events_auditory_run_3_sub_pp_03.mat renamed to sub-pp_03_audlocalizer_run-03_events.mat
events_auditory_run_3_sub_pp_03.tsv renamed to sub-pp_03_audlocalizer_run-03_events.tsv
events_visual_run_1_sub_pp_03.mat renamed to sub-pp_03_vislocalizer_events.mat
events_visual_run_1_sub_pp_03.tsv renamed to sub-pp_03_vislocalizer_events.tsv
pp_03_20260415131256_1.nii renamed to sub-pp_03_AAhead_scout_scout.nii
pp_03_20260415131256_10.nii renamed to sub-pp_03_T1w.nii
pp_03_20260415131256_11.nii renamed to sub-pp_03_acq-hipp_T2w.nii
pp_03_20260415131256_12.nii renamed to sub-pp_03_audlocalizer_run-03_bold.nii
pp_03_20260415131256_2.nii renamed to sub-pp_03_AAhead_scout_MPR_sag_scout.nii
pp_03_20260415131256_3.nii renamed to sub-pp_03_AAhead_scout_MPR_cor_scout.nii
pp_03_20260415131256_4.nii renamed to sub-pp_03_AAhead_scout

In [51]:
# This function moves all the files to the BIDS directory
# Any files not in bids go to /bids/sourcedata/sub-[]/other/
def move(subject_id, niidir, datadir, execute, verbose):
    subject_id = "sub-" + subject_id
    subject_dir = datadir + "/" + subject_id
    if execute:
        if os.path.exists(datadir):
            os.chdir(datadir)
        else:
            os.makedirs(datadir)
            os.chdir(datadir)

    maps = {'.mat': 'other', 'scout': 'other', 'practice': 'other', 'order': 'other', 'seq': 'other', 'audiotest': 'other', 'T1': 'anat',
            'T2': 'anat', 'run': 'func', 'bold': 'func', 'epi': 'fmap', 'vislocalizer_events.tsv': 'func'}

    for file in os.listdir(niidir):
        for val in maps.keys():
            if val in file:
                if val == 'run' and '.mat' in file: # Moves .mat events to other instead of func
                    dest = 'other'
                else:
                    dest = maps[val]
                if execute:
                    if not os.path.exists(f"{subject_dir}/{dest}"): os.makedirs(f"{subject_dir}/{dest}")
                    shutil.move(os.path.join(niidir, file), f"{subject_dir}/{dest}")
                if verbose: print(f"{file} sent to {dest}")
                break
    if execute:
        os.chdir(subject_dir)
        shutil.move(f"{subject_dir}/other", f"{datadir}/sourcedata/{subject_id}/other")
    else:
        if verbose: print(f"Would have moved {subject_dir}/other to {datadir}/sourcedata/{subject_id}/other")
    return 1

In [52]:
move("03", directory, bids_dir, False, True)

sub-pp_03_T1w.json sent to anat
sub-pp_03_AAhead_scout_MPR_cor_scout.nii sent to other
sub-pp_03_AAhead_scout_scout.json sent to other
sub-pp_03_audlocalizer_run-03_bold.nii sent to func
sub-pp_03_dir-AP_epi.json sent to fmap
sub-pp_03_AAhead_scout_MPR_sag_scout.json sent to other
sub-pp_03_audiotest_bold_13:48:18.832500.json sent to other
sub-pp_03_audlocalizer_run-01_bold.json sent to func
sub-pp_03_vislocalizer_events.tsv sent to func
sub-pp_03_acq-hipp_T2w.nii sent to anat
sub-pp_03_vislocalizer_bold.nii sent to func
sub-pp_03_AAhead_scout_MPR_cor_scout.json sent to other
sub-pp_03_dir-PA_epi.nii sent to fmap
sub-pp_03_audlocalizer_run-03_bold.json sent to func
sub-pp_03_audiotest_bold_13:48:18.832500.nii sent to other
sub-pp_03_audlocalizer_run-03_events.tsv sent to func
sub-pp_03_AAhead_scout_MPR_sag_scout.nii sent to other
sub-pp_03_audlocalizer_run-03_events.mat sent to other
sub-pp_03_acq-hipp_T2w.json sent to anat
sub-pp_03_dir-PA_epi.json sent to fmap
sub-pp_03_vislocalizer_

1

In [5]:
# This function formats JSON files for fieldmap files, functional files, and anatomical files
# Functional: Adds 'TaskName', deletes 'AcquisitionDuration'
# Anatomical: Deletes 'RepetitionTime'
# Fieldmaps: Adds 'IntendedFor', deletes 'RepetitionTime'
def format_json(subject_id, bids_dir, verbose):
    """
    This function formats JSON files for field map files, functional files, and anatomical files
    Functional: Adds 'TaskName', deletes 'AcquisitionDuration'
    Anatomical: Deletes 'RepetitionTime'
    Field maps: Adds 'IntendedFor', deletes 'RepetitionTime'
    """

    path_func = bids_dir + "/sub-" + subject_id + "/func"
    path_anat = bids_dir + "/sub-" + subject_id + "/anat"
    fmap_1 = bids_dir + "/sub-" + subject_id + "/fmap/" + "sub-" + subject_id + "_dir-AP_epi.json"
    fmap_2 = bids_dir + "/sub-" + subject_id + "/fmap/" + "sub-" + subject_id + "_dir-PA_epi.json"

    for file in os.listdir(path_func):
        if file.endswith(".nii"):
            # Edit the first fieldmap JSON. Add IntendedFor and delete repetition time
            with open(fmap_1, 'r') as json_file_1:
                json_data_1 = json.load(json_file_1)

            if 'IntendedFor' not in json_data_1: json_data_1['IntendedFor'] = []

            intended = f"func/{file}"
            if intended not in json_data_1['IntendedFor']:
                json_data_1['IntendedFor'].append(intended)

            if 'RepetitionTime' in json_data_1: del json_data_1['RepetitionTime']

            with open(fmap_1, 'w') as json_file_1:
                json.dump(json_data_1, json_file_1)

            # Edit the second fieldmap JSON. Add IntendedFor and delete repetition time
            with open(fmap_2, 'r') as json_file_2:
                json_data_2 = json.load(json_file_2)

            if 'IntendedFor' not in json_data_2: json_data_2['IntendedFor'] = []

            if intended not in json_data_2['IntendedFor']:
                json_data_2['IntendedFor'].append(intended)

            if 'RepetitionTime' in json_data_2: del json_data_2['RepetitionTime']

            with open(fmap_2, 'w') as json_file_2:
                json.dump(json_data_2, json_file_2)

            if verbose: print(f"Fieldmap jsons updated for {file}")
        elif file.endswith(".json"):

            # Check if this json is a functional one
            if 'bold' in file:
                # Edit the function JSON files. Add in TaskName and remove AcquisitionDuration
                with open(os.path.join(path_func, file), 'r') as func_json:
                    func_data = json.load(func_json)

                # Calculate the right task name
                task_name = func_data['SeriesDescription'].split('_')[0]

                # Add in task field
                func_data['TaskName'] = task_name

                # Delete acq duration
                if 'AcquisitionDuration' in func_data: del func_data['AcquisitionDuration']

                with open(os.path.join(path_func, file), 'w') as func_json:
                    json.dump(func_data, func_json)

                if verbose: print(f"Functional jsons updated for {file}")

    for file in os.listdir(path_anat):
        # Edit the anatomical data JSON files. Remove repetition time.
        if file.endswith(".json"):
            with open(os.path.join(path_anat, file), 'r') as anat_json:
                anat_data = json.load(anat_json)

            if 'RepetitionTime' in anat_data: del anat_data['RepetitionTime']

            with open(os.path.join(path_anat, file), 'w') as anat_json:
                json.dump(anat_data, anat_json)

            if verbose: print(f"Anatomical JSON edited for {file}")

    print("Formatting of JSONs is done")

    return 0

In [22]:
if execute: format_json(bids_dir)

Fieldmap jsons updated for sub-multimem002_task-multisensorymemory_run-2_bold.nii
Fieldmap jsons updated for sub-multimem002_task-multisensorymemory_run-7_bold.nii
Fieldmap jsons updated for sub-multimem002_task-multisensorymemory_run-3_bold.json
Fieldmap jsons updated for sub-multimem002_task-multisensorymemory_run-2_bold.json
Fieldmap jsons updated for sub-multimem002_task-multisensorymemory_run-1_bold.json
Fieldmap jsons updated for sub-multimem002_task-multisensorymemory_run-6_bold.nii
Fieldmap jsons updated for sub-multimem002_task-multisensorymemory_run-9_bold.json
Fieldmap jsons updated for sub-multimem002_task-multisensorymemory_run-4_events.tsv
Fieldmap jsons updated for sub-multimem002_task-multisensorymemory_run-4_bold.nii
Fieldmap jsons updated for sub-multimem002_task-multisensorymemory_run-6_bold.json
Fieldmap jsons updated for sub-multimem002_task-multisensorymemory_run-5_bold.json
Fieldmap jsons updated for sub-multimem002_task-multisensorymemory_run-8_bold.nii
Fieldmap

In [1]:
# Should be populated as soon as possible, so that we don't have to rerun everything later on
# levels is the categories variables can take
levels_dict = {'stimulus_name': {'category 1': 'some explanation', 'category 2': 'some explanation'}, 
               'event': {'category 1': 'some explanation', 'category 2': 'some explanation'}
              }

descriptions = {'onset': 'something', 'duration': 'something', 'tr': 'something', 'stimulus_name': 'something', 'event': 'something'}

In [30]:
def format_events(subject_id, bids_dir, descriptions, levels_dict, execute, verbose):
    """
    This function creates the required .json files for each .tsv events file which describe the column headers.

    Inputs:
    bids_dir: Path to the BIDS dataset
    descriptions: The description field as a dictionary for the .json file
    levels_dict: The levels field as a dictionary for the .json file

    Output:
    If execute = True, it will update the JSON event files
    Otherwise, it will print out path to JSON files and the data to be sent to them
    """

    path = bids_dir + "/sub-" + subject_id + "/func"

    for file in os.listdir(path):
        if file.endswith(".tsv"):
            # Loading events file
            temp = pd.read_csv(os.path.join(path, file), sep='\t')

            # Creating the required JSON files
            data_dict = {}
            for column in temp.columns:
                data_dict[column] = {"Description": f"{descriptions[column]}"}
                if column in levels_dict.keys(): data_dict[column]["Levels"] = levels_dict[column]

            json_filename = os.path.splitext(os.path.join(path, file))[0] + ".json"

            if execute:
                # Write a new .json file
                with open(json_filename, 'w') as f:
                    json.dump(data_dict, f)

                if verbose: print(f"Events file updated and JSON created for {file}")
            else:
                if verbose: print(json_filename)
                if verbose: print(data_dict)

    return 0


In [31]:
format_events(bids_dir, descriptions, levels_dict)

            onset  duration   tr stimulus_name                         event
0        0.027800  0.009415    0          None  TRCountdown and instructions
1        0.037215  0.008438    0          None  TRCountdown and instructions
2        0.045653  0.016891    0          None  TRCountdown and instructions
3        0.062544  0.016712    0          None  TRCountdown and instructions
4        0.079256  0.016523    0          None  TRCountdown and instructions
...           ...       ...  ...           ...                           ...
26484  502.679962  0.016726  326          None      TRCountdown and Feedback
26485  502.696688  0.016692  326          None      TRCountdown and Feedback
26486  502.713380  0.016689  326          None      TRCountdown and Feedback
26487  502.730069  0.001601  326          None      TRCountdown and Feedback
26488  502.731670  0.000000  327    End of Run                 Run completed

[26489 rows x 5 columns]


0

In [51]:
# Quick function to fix the slight problem in anatomical scan names
def fix_anat():
    path = bids_dir + subject_ID + "/anat"
    
    for file in os.listdir(path):
        if 'T2w' in file:
            new_file = f"{file.split('_')[0]}_{file.split('_')[1]}.{file.split('.')[-1]}"
            new_file = new_file.replace("anat-", "")
            if execute: os.rename(os.path.join(path, file), os.path.join(path, new_file))
            print(new_file)
        elif 'T1w' in file:
            new_file = file
            new_file = new_file.replace("anat-", "")
            if execute: os.rename(os.path.join(path, file), os.path.join(path, new_file))
            print(new_file)
         
    return 0


In [52]:
fix_anat()

sub-multimem002_T2w.json
sub-multimem002_T1w.nii
sub-multimem002_T1w.json
sub-multimem002_T2w.nii


0

Things to do:

1. ~Fix the column names of the events file~
2. ~Create code for accompanying JSON file generation of events file~
3. ~Execute events code for 002 subject data (which is already in BIDS)~
4. ~Figure out participants tsv file - NOT REQUIRED~
5. ~Figure out the dataset description file - REQUIRED~
6. ~Also needed: readme (required), citation (recommended), license (recommended), change log (optional)~
7. ~Comment out previous code~

~Change sub- as the folder name DONE~

~Change multimem-p002 to multimem002 DONE~

~Add .bidsignore for the other files (fixing by moving other to sourcedata) DONE~

~Change NaN value in events duration to 0's DONE~

~Rename anat folder files to remove anat in filenames DONE~

~Add task name to the json files for functional data DONE~

~remove acquisition duration for functional and repetition time for anatomical and fmap -> DONE~

~Fix the dataset_description field (upload new to milgram) DONE~